In [ ]:
import time

import mlflow
import pandas as pd
from dotenv import load_dotenv
from mlflow import MlflowClient
from mlflow.deployments import get_deploy_client

mlflow.set_registry_uri("databricks-uc")

In [ ]:
client_mlflow = MlflowClient()
client_databricks = get_deploy_client("databricks")

In [ ]:
endpoints = client_databricks.list_endpoints()
is_exist = any(e["name"] == "betsim" for e in endpoints)

if not is_exist:
    model_version = client_mlflow.get_model_version_by_alias(
        "betsim.models.xgboost_optuna",
        "champion",
    )

    endpoint = client_databricks.create_endpoint(
        config={
            "name": "betsim",
            "config": {
                "served_entities": [
                    {
                        "entity_name": "betsim.models.xgboost_optuna",
                        "entity_version": model_version.version,
                        "workload_size": "Small",
                        "workload_type": "CPU",
                        "scale_to_zero_enabled": True,
                    },
                ],
            },
        },
    )
    print("Endpoint created.")
else:
    endpoint = client_databricks.get_endpoint("betsim")
    print("Endpoint already exists.")

In [ ]:
while endpoint["state"]["ready"] != "READY":
    endpoint = client_databricks.get_endpoint("betsim")
    print("Waiting for endpoint to be ready...")
    time.sleep(10)

col = [
    "hcap_mag", "n_rest_day_net", "rating_seas_net",
    "rank_net", "scores_net", "rate_seas_lose_net",
    "rate_seas_win_net", "rating_hist_net", "xg_h2h_win",
    "xg_h2h_lose",
]
data = [0. for _ in col]

client_databricks.predict(
    endpoint="betsim",
    inputs={
        "dataframe_split": {
            "columns": col,
            "data": [data]
        },
    },
)